# Banking Intent Classifier – Live Prediction
Type a customer message and the saved model tells us the intent. No training happens here: the models are loaded from disk.

**Before running:** upload the `saved_model/` folder next to this notebook (in Colab: unzip `saved_model.zip` into a folder called `saved_model`).

In [9]:
from google.colab import files
_ = files.upload()

Saving saved_model (2).zip to saved_model (2).zip


In [10]:
!unzip -o saved_model.zip -d saved_model

Archive:  saved_model.zip
  inflating: saved_model/tfidf_logreg.joblib  
  inflating: saved_model/vocab.json  
  inflating: saved_model/lstm_model.keras  
  inflating: saved_model/classes.json  


## 1. Setup

In [11]:
import re, json, joblib
import numpy as np
from IPython.display import display, Markdown
from tensorflow import keras

SAVE_DIR = "saved_model"

# Same text preparation as in training. It must match the training code exactly.
TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:'[a-z]+)?|[^\sa-z0-9]")

def clean_text(text):
    return re.sub(r"\s+", " ", str(text).lower().strip())

def tokenize(text):
    return TOKEN_PATTERN.findall(clean_text(text))

## 2. Load the saved models, vocabulary and class names

In [12]:
tfidf_model = joblib.load(f"{SAVE_DIR}/tfidf_logreg.joblib")
lstm_model = keras.models.load_model(f"{SAVE_DIR}/lstm_model.keras")

with open(f"{SAVE_DIR}/vocab.json") as f:
    vocab = json.load(f)
with open(f"{SAVE_DIR}/classes.json") as f:
    CLASSES = json.load(f)

STOI = {tok: i for i, tok in enumerate(vocab["itos"])}
MAX_LEN = vocab["max_len"]
OOV_ID = 1

def encode(text):
    """Text -> padded id matrix of shape (1, MAX_LEN), same as in training."""
    ids = [STOI.get(tok, OOV_ID) for tok in tokenize(text)][:MAX_LEN]
    return np.array([ids + [0] * (MAX_LEN - len(ids))], dtype="int32")

print(f"Loaded 2 models, {len(vocab['itos'])} vocabulary tokens, {len(CLASSES)} intents.")

Loaded 2 models, 2205 vocabulary tokens, 77 intents.


## 3. Prediction function

In [13]:
def pretty(label):
    return label.replace("_", " ")

def show_prediction(text, top_k=3):
    tfidf_proba = tfidf_model.predict_proba([text])[0]
    lstm_proba = lstm_model.predict(encode(text), verbose=0)[0]

    best = int(tfidf_proba.argmax())
    lines = [
        f"**Message:** \"{text}\"",
        "",
        f"We predict this customer intent is **{pretty(CLASSES[best])}** "
        f"(confidence {tfidf_proba[best]:.0%}).",
        "",
        f"**Top {top_k} (TF-IDF + Logistic Regression, our best model):**",
    ]
    for rank, i in enumerate(tfidf_proba.argsort()[::-1][:top_k], 1):
        lines.append(f"{rank}. {pretty(CLASSES[i])} ({tfidf_proba[i]:.0%})")

    lstm_best = int(lstm_proba.argmax())
    lines += ["", f"*For comparison, the LSTM predicts* **{pretty(CLASSES[lstm_best])}** "
                  f"*({lstm_proba[lstm_best]:.0%}).*"]
    display(Markdown("\n".join(lines)))

## 4. Try it

In [14]:
text = input("Type a customer message: ")
show_prediction(text)

Type a customer message: Why was my transfer declined?


**Message:** "Why was my transfer declined?"

We predict this customer intent is **declined transfer** (confidence 88%).

**Top 3 (TF-IDF + Logistic Regression, our best model):**
1. declined transfer (88%)
2. failed transfer (8%)
3. declined cash withdrawal (1%)

*For comparison, the LSTM predicts* **declined transfer** *(96%).*

*Examples to try:* "I lost my card yesterday, please block it" · "Why was my transfer declined?" · "How long does a top up take?" · "My card has not arrived yet"